# Oil Well Location Optimization using Machine Learning

### Project Context
OilyGiant, an oil extraction company, needs to identify the optimal locations for drilling 200 new oil wells. This project develops a data-driven approach to maximize profitability while minimizing financial risks through regional analysis and predictive modeling.

### Business Problem
The company must strategically select drilling locations from three different regions, considering:
- **Budget constraint**: $100 million for 200 oil wells
- **Profitability threshold**: Each well must generate at least $500,000 to avoid losses
- **Risk management**: Accept only regions with loss probability < 2.5%

### Dataset Overview
- **Three geological regions** with exploration data (geo_data_0.csv, geo_data_1.csv, geo_data_2.csv)
- **Features**: f0, f1, f2 (geological characteristics of drilling points)
- **Target**: Product volume (oil reserves in thousands of barrels)
- **Sample size**: 500 points per region for analysis

### Project Objectives
1. **Develop predictive models** using linear regression for each region
2. **Identify top 200 drilling locations** with highest predicted reserves
3. **Calculate profitability** for each region (Revenue: $4.5 per barrel)
4. **Perform risk analysis** using bootstrapping technique (1,000 samples)
5. **Select optimal region** based on profit maximization and risk minimization

### Technical Approach
- **Model**: Linear Regression (as per business requirements)
- **Validation**: 75-25 train-test split with RMSE evaluation
- **Selection Strategy**: Top 200 wells by predicted reserves per region
- **Risk Assessment**: Bootstrap analysis with 95% confidence intervals
- **Decision Criteria**: Highest average profit with <2.5% loss probability

##### 1. Data Preparation

In [8]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

In [9]:
geo_data_0 = pd.read_csv('/datasets/geo_data_0.csv')
geo_data_1 = pd.read_csv('/datasets/geo_data_1.csv')
geo_data_2 = pd.read_csv('/datasets/geo_data_2.csv')

In [10]:
print(geo_data_1)

          id         f0         f1        f2     product
0      kBEdx -15.001348  -8.276000 -0.005876    3.179103
1      62mP7  14.272088  -3.475083  0.999183   26.953261
2      vyE1P   6.263187  -5.948386  5.001160  134.766305
3      KcrkZ -13.081196 -11.506057  4.999415  137.945408
4      AHL4O  12.702195  -8.147433  5.004363  134.766305
...      ...        ...        ...       ...         ...
99995  QywKC   9.535637  -6.878139  1.998296   53.906522
99996  ptvty -10.160631 -12.558096  5.005581  137.945408
99997  09gWa  -7.378891  -3.084104  4.998651  137.945408
99998  rqwUm   0.665714  -6.152593  1.000146   30.132364
99999  relB0  -3.426139  -7.794274 -0.003299    3.179103

[100000 rows x 5 columns]


The datasets for all three regions were successfully loaded and inspected. The relevant features and target variable were identified for model training.

##### 2. Model Training and Evaluation

The target variable for training and testing the model for 'geo_data_0.csv' is the 'product' column. The columns 'f0', 'f1', and 'f2' are the features used as independent variables

In [11]:
# Function to train and evaluate the model 
def test_region(data):
    target = data['product']
    features = data.drop(['id', 'product'], axis=1)

    # Splitting into training (75%) and validation (25%) sets
    features_train, features_valid, target_train, target_valid = train_test_split(
        features, target, test_size=0.25, random_state=12345)
    
    # Train the linear regression model
    model = LinearRegression()
    model.fit(features_train, target_train)
    predicted_valid = model.predict(features_valid) # predictions on validation set

    # Calculate and print evaluation metrics
    rmse = mean_squared_error(target_valid, predicted_valid, squared=False)
    print('Average volume of expected reserves:', predicted_valid.mean())
    print(f'RMSE: {rmse}\n')
    return predicted_valid, target_valid

# Evaluating each region
print('Region 0:')
preds_0 = test_region(geo_data_0)
print('Region 1:')
preds_1 = test_region(geo_data_1)
print('Region 2:')
preds_2 = test_region(geo_data_2)

Region 0:
Average volume of expected reserves: 92.59256778438035
RMSE: 37.5794217150813

Region 1:
Average volume of expected reserves: 68.728546895446
RMSE: 0.893099286775617

Region 2:
Average volume of expected reserves: 94.96504596800489
RMSE: 40.02970873393434



A linear regression model was trained for each region. The average predicted reserves and RMSE were calculated, showing reasonable accuracy for further analysis.

##### 3. Profit Calculation Preparation

In [12]:
unit_price = 4500 # Price in USD per unit (thousand barrels)
total_cost = 100000000 # # Total budget for 200 wells = 1M USD

#Function to calculate profit based on predictions
def calculate_profit(preds, unit_price, total_cost):
    selected_reserves = sorted(preds[0], reverse=True)[:200] # Select top 200 wells by predicted reserves
    total_selected_reserves = sum(selected_reserves) # Sum all reserves from selected wells
    total_profit = total_selected_reserves * unit_price - total_cost # Calculate total profit
    return total_profit

# Calculate profit for each region using the predictions
total_profit_0 = calculate_profit(preds_0, unit_price, total_cost)
total_profit_1 = calculate_profit(preds_1, unit_price, total_cost)
total_profit_2 = calculate_profit(preds_2, unit_price, total_cost)

# Evaluating each region
print('Region 0:')
print(f'Total profit: ${total_profit_0:,.2f}\n')
print('Region 1:')
print(f'Total profit: ${total_profit_1:,.2f}\n')
print('Region 2:')
print(f'Total profit: ${total_profit_2:,.2f}')

Region 0:
Total profit: $39,960,488.77

Region 1:
Total profit: $24,857,120.52

Region 2:
Total profit: $33,217,543.96


All the needed values for profit calculation were set. The average predicted reserves for each well were checked against the break-even point. This shows that the investment is likely to be profitable.

##### 4. Profit Calculation for Top Wells

In [13]:
def profit_top_wells(predictions, unit_price=4500, total_cost=100000000, wells=200):
    top_reserves = sorted(predictions, reverse=True)[:wells] # Sort predictions in descending order and select top 'wells' number
    total_reserves = sum(top_reserves) # Calculate total reserves by summing all selected wells
    profit = total_reserves * unit_price - total_cost # Calculate final profit
    return profit, total_reserves 

# Apply function to Regions 0, 1 and 2
profit_0, reserves_0 = profit_top_wells(preds_0[0]) #returns a tuple 'profit' and 'total_reserves'
profit_1, reserves_1 = profit_top_wells(preds_1[0])
profit_2, reserves_2 = profit_top_wells(preds_2[0])

# Evaluating each region
print('Region 0:')
print(f'| Profit=${profit_0:,.2f} | Reserves={reserves_0:,.2f} |\n')
print('Region 1:')
print(f'| Profit=${profit_1:,.2f} | Reserves={reserves_1:,.2f} |\n')
print('Region 2:')
print(f'| Profit=${profit_2:,.2f} | Reserves={reserves_2:,.2f} |')

Region 0:
| Profit=$39,960,488.77 | Reserves=31,102.33 |

Region 1:
| Profit=$24,857,120.52 | Reserves=27,746.03 |

Region 2:
| Profit=$33,217,543.96 | Reserves=29,603.90 |


The top 200 wells with the highest predicted reserves were selected for each region. The total profit and reserves were calculated, allowing for a comparison between regions.

##### 5. Risk and Profit Analysis (Bootstrapping)

In [14]:
def revenue(predictions, unit_price=4500, total_cost=100000000, count=200):
    selected = sorted(predictions, reverse=True)[:count] # Sort predictions and select the top 200 wells
    return sum(selected) * unit_price - total_cost # Calculate total profit

def bootstrap_profit(predictions, unit_price=4500, total_cost=100000000, wells=200, iterations=1000, sample_size=500): # sample: 500
    preds = pd.Series(predictions)  # Convert predictions to 'Series' for sampling

    state = np.random.RandomState(12345)

    values = []
    for i in range(iterations):
        sample = preds.sample(n=sample_size, replace=True, random_state=state) # Exploration of 500 wells (with replacement)
        profit = revenue(sample, unit_price, total_cost, wells) # Calculate profit only for the top 200 wells from the 500
        values.append(profit)

    values = pd.Series(values)

    # Metrics
    mean = values.mean()  # Average profit
    lower = values.quantile(0.025)  # Lower 
    upper = values.quantile(0.975)  # Upper
    risk = (values < 0).mean() * 100  # Probability of loss
    return mean, (lower, upper), risk

# Apply bootstrapping to each region
mean_0, conf_0, risk_0 = bootstrap_profit(preds_0[0])
mean_1, conf_1, risk_1 = bootstrap_profit(preds_1[0])
mean_2, conf_2, risk_2 = bootstrap_profit(preds_2[0])

# Evaluating each region
print('Region 0:')
print(f'| Mean=${mean_0:,.2f} | 95% CI={conf_0} | Risk={risk_0:.2f}% |\n')
print('Region 1:')
print(f'| Mean=${mean_1:,.2f} | 95% CI={conf_1} | Risk={risk_1:.2f}% |\n')
print('Region 2:')
print(f'| Mean=${mean_2:,.2f} | 95% CI={conf_2} | Risk={risk_2:.2f}% |')

Region 0:
| Mean=$3,580,260.54 | 95% CI=(1401373.35981073, 5988772.713793854) | Risk=0.10% |

Region 1:
| Mean=$4,538,116.10 | 95% CI=(328830.1962251447, 8545616.85213055) | Risk=1.60% |

Region 2:
| Mean=$2,814,661.98 | 95% CI=(920983.0189543873, 4785270.324733695) | Risk=0.20% |


Bootstrapping was performed to estimate the profit distribution, 95% confidence interval, and risk of loss for each region. The analysis identified the region with the highest average profit and lowest risk.

##### Final Conclusion

After evaluating all three regions using linear regression, profit calculations, and bootstrapping for risk analysis, it was found that all regions present less than 2.5% risk of loss when selecting the top 200 wells. However, Region 1 stands out by offering the highest mean profit among the three.